# 04 — Feature Engineering

**Wind Turbine Predictive Maintenance & Failure Intelligence System**

## Objective
Turn raw cleaned sensors into features that expose degradation. NB02 showed the
signal is subtle and fault-specific — raw values alone won't separate pre-fault
from normal. This notebook builds temporal and domain features that capture
*how sensors are behaving over time*, not just their instantaneous values.

## Feature families built here
1. **Lag features** — sensor values N steps back.
2. **Rolling statistics** — backward-only rolling mean/std/min/max.
3. **Rate-of-change** — how fast a sensor is moving.
4. **Domain ratios & deltas** — bearing−ambient, gearbox−ambient, power/wind (later cells).
5. **Operating-state** — running indicator (later cells).

## The leakage rule (non-negotiable)
- All features computed **per dataset, in time order, backward-looking only**.
- A feature at row *t* uses only rows *≤ t* — never future rows.
- Features never cross dataset-file boundaries (each file = one continuous run).
- `status_type_id` is excluded from features (NB02: it's entangled with the fault → leakage).

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

BASE = Path("..") / "data" / "raw" / "Wind Farm A"
PROCESSED_DIR = Path("..") / "data" / "processed"
FEATURES_DIR = Path("..") / "data" / "processed" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

events = pd.read_csv(BASE / "event_info.csv", sep=";")
clean_files = sorted(PROCESSED_DIR.glob("*_clean.csv"), key=lambda p: int(p.stem.split("_")[0]))
print(f"Cleaned files found: {len(clean_files)}")

# Load one to work out the feature logic before applying to all
df = pd.read_csv(clean_files[0])
df["time_stamp"] = pd.to_datetime(df["time_stamp"])
df = df.sort_values("id").reset_index(drop=True)
print("Sample file:", clean_files[0].name, "shape:", df.shape)

# Identify the sensor columns we'll engineer from (exclude metadata + leakage cols)
META = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]
sensor_cols = [c for c in df.columns if c not in META]
print(f"Sensor columns available: {len(sensor_cols)}")

Cleaned files found: 22
Sample file: 0_clean.csv shape: (54986, 94)
Sensor columns available: 89


## 1. Lag features

A lag feature gives the model the sensor value from N steps earlier, so each row
carries recent history. We lag the **key condition sensors** (bearing, gearbox,
generator, transformer, hydraulic temps + power/RPM), not all 89 columns — with
only 12 events, a huge feature count would overfit.

Lags: 1, 3, 6 steps (10, 30, 60 minutes back). Computed in time order per file;
the first N rows get NaN (no history) and are handled at the end.

In [2]:
# Key sensors to build temporal features from (from NB02 domain analysis)
KEY_SENSORS = [
    "sensor_11_avg",  # gearbox bearing HS temp
    "sensor_12_avg",  # gearbox oil temp
    "sensor_13_avg",  # generator bearing DE temp
    "sensor_14_avg",  # generator bearing NDE temp
    "sensor_38_avg",  # HV transformer L1 temp
    "sensor_41_avg",  # hydraulic oil temp
    "sensor_0_avg",   # ambient temp
    "power_30_avg",   # grid power
    "sensor_18_avg",  # generator RPM
    "sensor_52_avg",  # rotor RPM
    "wind_speed_3_avg",  # wind speed
]
print(f"Building temporal features from {len(KEY_SENSORS)} key sensors")

LAGS = [1, 3, 6]  # 10, 30, 60 minutes

def add_lag_features(df, cols, lags):
    """Add lagged versions of cols. Assumes df is sorted in time order (one file)."""
    df = df.sort_values("id").reset_index(drop=True)
    new = {}
    for col in cols:
        for lag in lags:
            new[f"{col}_lag{lag}"] = df[col].shift(lag)
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

# Test on the sample file
df_lagged = add_lag_features(df, KEY_SENSORS, LAGS)
new_cols = [c for c in df_lagged.columns if "_lag" in c]
print(f"Added {len(new_cols)} lag features")
print("Example:", new_cols[:6])

# Sanity check: lag1 of a column should equal the original shifted down by 1
check = pd.DataFrame({
    "original": df["sensor_13_avg"].head(5),
    "lag1": df_lagged["sensor_13_avg_lag1"].head(5),
    "lag3": df_lagged["sensor_13_avg_lag3"].head(5),
})
print("\nLag sanity check (lag1 should be original shifted down 1 row):")
print(check)

Building temporal features from 11 key sensors
Added 33 lag features
Example: ['sensor_11_avg_lag1', 'sensor_11_avg_lag3', 'sensor_11_avg_lag6', 'sensor_12_avg_lag1', 'sensor_12_avg_lag3', 'sensor_12_avg_lag6']

Lag sanity check (lag1 should be original shifted down 1 row):
   original  lag1  lag3
0      32.0   NaN   NaN
1      32.0  32.0   NaN
2      32.0  32.0   NaN
3      32.0  32.0  32.0
4      31.0  32.0  32.0


## 2. Rolling statistics

The single most important feature family. NB02 showed the pre-fault signal lives
in *sustained* elevation and increased variability — not raw spiky values. Rolling
statistics capture this:
- **rolling mean** — smooths transient spikes, exposes sustained drift.
- **rolling std** — captures increasing erraticness / instability.
- **rolling min & max** — the recent envelope of operation.

Windows: 6, 36, 144 steps (1 h, 6 h, 24 h). All windows are **backward-looking**
(pandas `.rolling()` default), so row *t* summarises only rows *≤ t* — no leakage.
Computed per file so windows never cross run boundaries.

In [3]:
WINDOWS = [6, 36, 144]  # 1 hour, 6 hours, 24 hours (at 10-min sampling)

def add_rolling_features(df, cols, windows):
    """Add backward-looking rolling mean/std/min/max. Assumes time-sorted (one file)."""
    df = df.sort_values("id").reset_index(drop=True)
    new = {}
    for col in cols:
        for w in windows:
            roll = df[col].rolling(window=w, min_periods=max(2, w // 4))
            new[f"{col}_rmean{w}"] = roll.mean()
            new[f"{col}_rstd{w}"]  = roll.std()
            new[f"{col}_rmin{w}"]  = roll.min()
            new[f"{col}_rmax{w}"]  = roll.max()
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

# Test on the sample (already has lags from cell 4)
df_roll = add_rolling_features(df_lagged, KEY_SENSORS, WINDOWS)
roll_cols = [c for c in df_roll.columns if any(f"_r{s}" in c for s in ["mean","std","min","max"])]
print(f"Added {len(roll_cols)} rolling features")
print("Example:", [c for c in roll_cols if c.startswith("sensor_13_avg")][:6])

# Sanity check: rolling mean at row t must equal mean of the window ENDING at t (backward)
w = 6
manual = df["sensor_13_avg"].iloc[max(0,10-w+1):11].mean()  # rows 5..10 inclusive
auto = df_roll["sensor_13_avg_rmean6"].iloc[10]
print(f"\nRolling sanity check (row 10, window 6):")
print(f"  manual mean of rows 5-10: {manual:.3f}")
print(f"  rolling rmean6 at row 10: {auto:.3f}")
print(f"  match: {np.isclose(manual, auto)}")

Added 132 rolling features
Example: ['sensor_13_avg_rmean6', 'sensor_13_avg_rstd6', 'sensor_13_avg_rmin6', 'sensor_13_avg_rmax6', 'sensor_13_avg_rmean36', 'sensor_13_avg_rstd36']

Rolling sanity check (row 10, window 6):
  manual mean of rows 5-10: 31.000
  rolling rmean6 at row 10: 31.000
  match: True


## 3. Rate-of-change features

Lags and rolling means capture level and trend; rate-of-change captures *speed* —
how fast a sensor is moving. Degradation often appears as acceleration (a temp
climbing faster than normal) before the absolute value looks alarming. Two forms:
- **diff** — change over N steps (short-term velocity).
- **rolling-mean slope** — change in the smoothed signal, less noisy.

All backward-looking (differences of current vs earlier rows), per file.

In [4]:
ROC_STEPS = [3, 36]  # 30 min (fast), 6 h (smoothed trend)

def add_roc_features(df, cols, steps):
    """Backward-looking rate-of-change features. Assumes time-sorted (one file)."""
    df = df.sort_values("id").reset_index(drop=True)
    new = {}
    for col in cols:
        for s in steps:
            # raw difference over s steps (current minus s-steps-ago)
            new[f"{col}_diff{s}"] = df[col] - df[col].shift(s)
            # slope of the smoothed signal (change in rolling mean over s steps)
            rmean = df[col].rolling(window=s, min_periods=max(2, s // 4)).mean()
            new[f"{col}_slope{s}"] = rmean - rmean.shift(s)
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

df_roc = add_roc_features(df_roll, KEY_SENSORS, ROC_STEPS)
roc_cols = [c for c in df_roc.columns if "_diff" in c or "_slope" in c]
print(f"Added {len(roc_cols)} rate-of-change features")
print("Example:", [c for c in roc_cols if c.startswith("sensor_13_avg")])

# Sanity check: diff3 at row t = value[t] - value[t-3]
t = 10
manual = df["sensor_13_avg"].iloc[t] - df["sensor_13_avg"].iloc[t-3]
auto = df_roc["sensor_13_avg_diff3"].iloc[t]
print(f"\nROC sanity check (row {t}, diff3):")
print(f"  manual value[{t}] - value[{t-3}]: {manual:.3f}")
print(f"  auto diff3 at row {t}: {auto:.3f}")
print(f"  match: {np.isclose(manual, auto)}")

print(f"\nTotal features on sample so far: {df_roc.shape[1]} columns")

Added 44 rate-of-change features
Example: ['sensor_13_avg_diff3', 'sensor_13_avg_slope3', 'sensor_13_avg_diff36', 'sensor_13_avg_slope36']

ROC sanity check (row 10, diff3):
  manual value[10] - value[7]: 0.000
  auto diff3 at row 10: 0.000
  match: True

Total features on sample so far: 303 columns


## 4. Domain ratios & deltas

Physically-motivated features from turbine engineering, acting on the NB02 finding
that fault signal appears as *temperature above ambient*, not raw temperature:
- **Temperature deltas** (component − ambient): isolates abnormal heating from
  weather. A hot bearing on a hot day is normal; a hot bearing on a cold day is not.
- **Component-vs-component deltas**: e.g. the two generator bearings should track
  each other — a growing gap signals one degrading.
- **Power-to-wind ratio**: efficiency indicator — low power at high wind suggests a
  problem. Guarded against divide-by-zero.

These are computed row-wise (no time window), so no leakage concern — but still
per file for consistency.

In [5]:
def add_domain_features(df):
    """Physically-motivated ratio/delta features. Row-wise, no time dependency."""
    df = df.copy()
    amb = df["sensor_0_avg"]  # ambient temp

    # Temperature deltas above ambient (abnormal heating signal)
    for col, name in [
        ("sensor_11_avg", "gearbox_bearing"),
        ("sensor_12_avg", "gearbox_oil"),
        ("sensor_13_avg", "gen_bearing_de"),
        ("sensor_14_avg", "gen_bearing_nde"),
        ("sensor_38_avg", "transformer"),
        ("sensor_41_avg", "hydraulic_oil"),
    ]:
        df[f"delta_{name}_amb"] = df[col] - amb

    # Component-vs-component: the two generator bearings should track together
    df["delta_gen_bearings"] = df["sensor_13_avg"] - df["sensor_14_avg"]

    # Gearbox bearing vs gearbox oil (both should rise together under load)
    df["delta_gearbox_bear_oil"] = df["sensor_11_avg"] - df["sensor_12_avg"]

    # Power-to-wind efficiency ratio (guard divide-by-zero)
    df["power_per_wind"] = df["power_30_avg"] / (df["wind_speed_3_avg"] + 0.1)

    return df

df_dom = add_domain_features(df_roc)
domain_cols = [c for c in df_dom.columns if c.startswith("delta_") or c == "power_per_wind"]
print(f"Added {len(domain_cols)} domain features:")
for c in domain_cols:
    print("  ", c)

# Quick sanity: gen-bearing delta-vs-ambient should be positive when running (bearings hotter than air)
running = df_dom[df_dom["power_30_avg"] > 0.05]
print(f"\nMean delta_gen_bearing_de_amb when running: "
      f"{running['delta_gen_bearing_de_amb'].mean():.1f} °C (should be clearly positive)")

Added 9 domain features:
   delta_gearbox_bearing_amb
   delta_gearbox_oil_amb
   delta_gen_bearing_de_amb
   delta_gen_bearing_nde_amb
   delta_transformer_amb
   delta_hydraulic_oil_amb
   delta_gen_bearings
   delta_gearbox_bear_oil
   power_per_wind

Mean delta_gen_bearing_de_amb when running: 35.0 °C (should be clearly positive)


## 5. Operating-state features

NB02's bimodal power/RPM distributions showed turbines are either idle or running
near capacity. Sensor readings mean different things in each state — a rising
bearing temp matters while producing, not while parked. These features make
operating state explicit so the model separates "abnormal while running" from
"normal idle", rather than conflating them:
- **is_running** — producing meaningful power.
- **is_ramping** — partial power (starting up / winding down / curtailed).
- **fraction of recent time running** — captures unstable start/stop cycling.

In [7]:
def add_operating_state_features(df):
    """Operating-state indicators. is_running/is_ramping are row-wise;
    the running-fraction is a backward rolling feature (leakage-safe)."""
    df = df.sort_values("id").reset_index(drop=True).copy()

    p = df["power_30_avg"]
    # Row-wise state flags (power is normalised 0..1 in this dataset)
    df["is_running"] = (p > 0.05).astype(int)
    df["is_ramping"] = ((p > 0.05) & (p < 0.85)).astype(int)

    # Backward rolling: fraction of the last 36 steps (6h) spent running
    df["frac_running_6h"] = (
        df["is_running"].rolling(window=36, min_periods=6).mean()
    )
    # Backward rolling: number of start/stop transitions in last 6h (cycling stress)
    transitions = df["is_running"].diff().abs()
    df["n_transitions_6h"] = transitions.rolling(window=36, min_periods=6).sum()

    return df

df_op = add_operating_state_features(df_dom)
op_cols = ["is_running", "is_ramping", "frac_running_6h", "n_transitions_6h"]
print("Added operating-state features:", op_cols)
print(f"\nFraction of rows running: {df_op['is_running'].mean():.1%}")
print(f"Fraction of rows ramping: {df_op['is_ramping'].mean():.1%}")

# Sanity: frac_running_6h at a row = mean of is_running over the prior window (backward)
t = 100
manual = df_op["is_running"].iloc[max(0,t-36+1):t+1].mean()
auto = df_op["frac_running_6h"].iloc[t]
print(f"\nSanity (row {t}): manual frac={manual:.3f}, feature frac={auto:.3f}, "
      f"match={np.isclose(manual, auto)}")

print(f"\nTotal columns on sample now: {df_op.shape[1]}")

Added operating-state features: ['is_running', 'is_ramping', 'frac_running_6h', 'n_transitions_6h']

Fraction of rows running: 59.3%
Fraction of rows ramping: 44.3%

Sanity (row 100): manual frac=0.806, feature frac=0.806, match=True

Total columns on sample now: 316
